In [ ]:
from tree_sitter_language_pack import get_language, get_parser
from html.parser import HTMLParser
from typing import Optional
from pathlib import Path
import tiktoken
import pandas as pd
import re

parser = get_parser("javascript")
enc = tiktoken.get_encoding("cl100k_base")

REPO_PATH = Path("/Users/robertandersson/dev/platoscave")

ignore_dirs = {
    "node_modules", ".git", "dist", "build",
    ".wrangler", ".next", ".vscode", "coverage",
    ".venv", "venv", "__pycache__",
    "vendor",
}

In [ ]:
def _text(node, source: bytes) -> str:
    return source[node.start_byte:node.end_byte].decode("utf-8", errors="replace")


def _clean_comment(raw: str) -> str:
    """Normalise // single-line and /* block */ comments to plain text."""
    text = raw.strip()
    if text.startswith("//"):
        return text[2:].strip()
    if text.startswith("/*"):
        text = text[2:]
    if text.endswith("*/"):
        text = text[:-2]
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("*"):
            line = line[1:].strip()
        if line:
            lines.append(line)
    return " ".join(lines)


def _looks_like_section_divider(text: str) -> bool:
    if not text:
        return True
    letters = sum(1 for c in text if c.isalpha())
    return letters / len(text) < 0.4


def _leading_doc_comment(tree, source: bytes) -> str:
    """Return the first meaningful leading comment, or '' if absent."""
    for node in tree.root_node.children[:5]:
        if node.type == "hash_bang_line":
            continue
        if node.type == "expression_statement":
            if _text(node, source).strip() in ("'use strict';", '"use strict";'):
                continue
            return ""
        if node.type == "comment":
            cleaned = _clean_comment(_text(node, source))
            return "" if _looks_like_section_divider(cleaned) else cleaned
        return ""
    return ""


def _from_function_decl(node, source: bytes) -> dict:
    name_node = node.child_by_field_name("name")
    params_node = node.child_by_field_name("parameters")
    return {
        "kind": "function",
        "name": _text(name_node, source) if name_node else "<anon>",
        "signature": _text(params_node, source) if params_node else "()",
        "line": node.start_point[0] + 1,
    }


def _from_lexical_decl(node, source: bytes) -> Optional[dict]:
    """Keep const foo = () => ... and const foo = function() {}, skip plain constants."""
    declarator = next((c for c in node.children if c.type == "variable_declarator"), None)
    if not declarator:
        return None
    name_node = declarator.child_by_field_name("name")
    value_node = declarator.child_by_field_name("value")
    if not name_node or not value_node:
        return None
    if value_node.type not in ("arrow_function", "function_expression"):
        return None
    params_node = value_node.child_by_field_name("parameters")
    return {
        "kind": "const_function",
        "name": _text(name_node, source),
        "signature": _text(params_node, source) if params_node else "()",
        "line": node.start_point[0] + 1,
    }


def extract_symbols(tree, source: bytes) -> list:
    """Extract top-level named function and const-function declarations."""
    symbols = []
    for node in tree.root_node.children:
        if node.type == "function_declaration":
            symbols.append(_from_function_decl(node, source))
        elif node.type == "lexical_declaration":
            sym = _from_lexical_decl(node, source)
            if sym:
                symbols.append(sym)
    return symbols

In [ ]:
class HtmlSurfaceParser(HTMLParser):
    """Extract title, first h1, and script/stylesheet src lists."""

    def __init__(self):
        super().__init__()
        self.title = None
        self.h1 = None
        self.scripts = []
        self._current_tag = None
        self._capture_buffer = []

    def handle_starttag(self, tag, attrs):
        attrs_dict = dict(attrs)
        if tag == "title" and self.title is None:
            self._current_tag = "title"
            self._capture_buffer = []
        elif tag == "h1" and self.h1 is None:
            self._current_tag = "h1"
            self._capture_buffer = []
        elif tag == "script" and "src" in attrs_dict:
            self.scripts.append(attrs_dict["src"])

    def handle_endtag(self, tag):
        if tag == self._current_tag:
            captured = re.sub(r"\s+", " ", " ".join(self._capture_buffer)).strip()
            setattr(self, tag, captured)
            self._current_tag = None
            self._capture_buffer = []

    def handle_data(self, data):
        if self._current_tag:
            self._capture_buffer.append(data)


def extract_html_metadata(path: Path) -> dict:
    p = HtmlSurfaceParser()
    try:
        p.feed(path.read_text(encoding="utf-8", errors="replace"))
    except Exception:
        pass

    def normalize(src: str) -> str:
        src = src.split("?")[0].split("#")[0]
        return src[2:] if src.startswith("./") else src

    scripts = [normalize(s) for s in p.scripts]
    return {
        "title": p.title or "",
        "h1": p.h1 or "",
        "purpose": p.title or p.h1 or path.stem.replace("-", " "),
        "local_scripts": [s for s in scripts if not s.startswith(("http://", "https://", "//"))],
        "external_scripts": [s for s in scripts if s.startswith(("http://", "https://", "//"))],
    }

In [ ]:
def is_minified(path: Path) -> bool:
    return ".min." in path.name

def should_process_js(path: Path) -> bool:
    return path.is_file() and path.suffix == ".js" and not is_minified(path) \
        and not any(part in ignore_dirs for part in path.parts)

def should_process_html(path: Path) -> bool:
    return path.is_file() and path.suffix == ".html" \
        and not any(part in ignore_dirs for part in path.parts)

def classify_group(path: Path) -> str:
    parts = path.relative_to(REPO_PATH).parts
    return "root" if len(parts) == 1 else parts[0]


# ── JS ─────────────────────────────────────────────────────────────
js_records, js_errors = [], []

for path in REPO_PATH.rglob("*.js"):
    if not should_process_js(path):
        continue
    try:
        source = path.read_bytes()
        tree = parser.parse(source)
        syms = extract_symbols(tree, source)
        js_records.append({
            "path": str(path.relative_to(REPO_PATH)),
            "group": classify_group(path),
            "purpose": _leading_doc_comment(tree, source),
            "symbols": syms,
            "n_symbols": len(syms),
            "tokens": len(enc.encode(source.decode("utf-8", errors="replace"))),
            "lines": source.count(b"\n") + 1,
        })
    except Exception as e:
        js_errors.append((str(path), str(e)))

repo_df = pd.DataFrame(js_records)


# ── HTML ───────────────────────────────────────────────────────────
html_records = []

for path in REPO_PATH.rglob("*.html"):
    if not should_process_html(path):
        continue
    try:
        meta = extract_html_metadata(path)
        source_bytes = path.read_bytes()
        html_records.append({
            "path": str(path.relative_to(REPO_PATH)),
            "group": classify_group(path),
            "purpose": meta["purpose"],
            "title": meta["title"],
            "h1": meta["h1"],
            "local_scripts": meta["local_scripts"],
            "external_scripts": meta["external_scripts"],
            "n_scripts": len(meta["local_scripts"]),
            "tokens": len(enc.encode(source_bytes.decode("utf-8", errors="replace"))),
            "lines": source_bytes.count(b"\n") + 1,
        })
    except Exception as e:
        print(f"⚠️  {path.relative_to(REPO_PATH)}: {e}")

html_df = pd.DataFrame(html_records)


# ── Summary ─────────────────────────────────────────────────────────
print(f"JS: {len(repo_df)} files, {len(js_errors)} errors")
if js_errors:
    for p, e in js_errors[:5]:
        print(f"  ⚠️  {p}: {e}")
print(repo_df.groupby("group").agg(
    files=("path", "count"), symbols=("n_symbols", "sum"), tokens=("tokens", "sum")
).sort_values("tokens", ascending=False).to_string())

missing = repo_df[repo_df["purpose"] == ""]
print(f"\nMissing purpose comments: {len(missing)} of {len(repo_df)}")
for p in missing["path"]:
    print(f"  {p}")

print(f"\nHTML: {len(html_df)} files")
print(html_df.groupby("group").agg(
    files=("path", "count"), scripts=("n_scripts", "sum"), tokens=("tokens", "sum")
).sort_values("tokens", ascending=False).to_string())

In [ ]:
OUTPUT_PATH = REPO_PATH / "REPO_MAP.md"

JS_GROUP_ORDER  = ["modules", "cases", "scripts", "tests", "js", "root"]
JS_GROUP_LABELS = {
    "modules": "Modules", "cases": "Cases", "scripts": "Build scripts",
    "tests": "Contract tests", "js": "Page-level scripts", "root": "Root-level files",
}
HTML_GROUP_ORDER  = ["articles", "cases", "modules", "root"]
HTML_GROUP_LABELS = {
    "articles": "Articles", "cases": "Case pages",
    "modules": "Module pages", "root": "Root-level pages",
}


def collapse_ws(s: str) -> str:
    return " ".join(s.split())

def render_js_file(row) -> str:
    lines = [f"### `{row['path']}`"]
    if row["purpose"]:
        lines.append(row["purpose"])
    lines.append(f"*{row['n_symbols']} symbols · {row['lines']} lines · {row['tokens']:,} tokens*")
    if row["symbols"]:
        lines.append("")
        for sym in row["symbols"]:
            lines.append(f"  - `{sym['name']}{collapse_ws(sym['signature'])}` (L{sym['line']})")
    return "\n".join(lines)

def render_html_file(row) -> str:
    lines = [f"### `{row['path']}`"]
    if row["title"]:
        lines.append(f"**Title:** {row['title']}")
    if row["h1"] and row["h1"] != row["title"]:
        lines.append(f"**H1:** {row['h1']}")
    lines.append(f"*{row['lines']} lines · {row['tokens']:,} tokens*")
    if row["local_scripts"]:
        lines += ["", "Scripts loaded:"] + [f"  - `{s}`" for s in row["local_scripts"]]
    if row["external_scripts"]:
        lines += ["", "External scripts:"] + [f"  - `{s}`" for s in row["external_scripts"]]
    return "\n".join(lines)

def render_js_group(g: str, df_g) -> str:
    label = JS_GROUP_LABELS.get(g, g.title())
    parts = [f"### {label}",
             f"*{len(df_g)} files · {df_g['n_symbols'].sum()} symbols · {df_g['tokens'].sum():,} tokens*", ""]
    for _, row in df_g.sort_values("path").iterrows():
        parts += [render_js_file(row), ""]
    return "\n".join(parts)

def render_html_group(g: str, df_g) -> str:
    label = HTML_GROUP_LABELS.get(g, g.title())
    parts = [f"### {label}",
             f"*{len(df_g)} pages · {df_g['n_scripts'].sum()} script refs · {df_g['tokens'].sum():,} tokens*", ""]
    for _, row in df_g.sort_values("path").iterrows():
        parts += [render_html_file(row), ""]
    return "\n".join(parts)


all_js_groups   = JS_GROUP_ORDER   + [g for g in repo_df["group"].unique()  if g not in JS_GROUP_ORDER]
all_html_groups = HTML_GROUP_ORDER + [g for g in html_df["group"].unique() if g not in HTML_GROUP_ORDER]

doc = [
    "# platoscave — Repository Map", "",
    f"*Auto-generated. {len(repo_df)} JS files · {repo_df['n_symbols'].sum()} symbols · "
    f"{len(html_df)} HTML pages · {repo_df['tokens'].sum() + html_df['tokens'].sum():,} total tokens. "
    f"Excludes vendor and minified files.*", "",
    "This is a structural index of the repository. JS files list top-level "
    "functions; HTML pages list which scripts they load. Use this to orient before reading source.", "",
    "## Overview", "", "### JavaScript", "",
    "| Group | Files | Symbols | Tokens |", "|-------|------:|--------:|-------:|",
]
for g in all_js_groups:
    sub = repo_df[repo_df["group"] == g]
    if len(sub):
        doc.append(f"| {JS_GROUP_LABELS.get(g, g)} | {len(sub)} | {sub['n_symbols'].sum()} | {sub['tokens'].sum():,} |")

doc += ["", "### HTML pages", "", "| Group | Pages | Script refs | Tokens |", "|-------|------:|------------:|-------:|"]
for g in all_html_groups:
    sub = html_df[html_df["group"] == g]
    if len(sub):
        doc.append(f"| {HTML_GROUP_LABELS.get(g, g)} | {len(sub)} | {sub['n_scripts'].sum()} | {sub['tokens'].sum():,} |")

doc += ["", "## JavaScript", ""]
for g in all_js_groups:
    sub = repo_df[repo_df["group"] == g]
    if len(sub):
        doc.append(render_js_group(g, sub))

doc += ["## Pages", ""]
for g in all_html_groups:
    sub = html_df[html_df["group"] == g]
    if len(sub):
        doc.append(render_html_group(g, sub))

OUTPUT_PATH.write_text("\n".join(doc), encoding="utf-8")
print(f"Wrote {OUTPUT_PATH}  ({OUTPUT_PATH.stat().st_size:,} bytes, {OUTPUT_PATH.read_text().count(chr(10)) + 1:,} lines)")